# 06 – Auswertung der Themen

1. Die finalen Modelle trainieren: LDA (gensim) auf Bag-of-Words, NMF (scikit-learn) auf TF-IDF
2. Die zehn wichtigsten Wörter je Thema ausgeben und prüfen, ob die Modelle mit `05_themenanzahl` übereinstimmen
3. Jeder Beschwerde ihr Thema zuordnen 
4. LDA und NMF vergleichen

In [7]:
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
K_LDA = 12
K_NMF = 15 
RANDOM_STATE = 42
LDA_PASSES = 10
NMF_MAX_ITER = 500
TOPN = 10

path = Path("data/tokens_final.pkl")
if not path.exists():
    raise FileNotFoundError("data/tokens_final.pkl fehlt, bitte zuerst 02_vorverarbeitung ausführen.")
df = pd.read_pickle(path)

# Vektorisierung 
tv = TfidfVectorizer(analyzer=lambda tokens: tokens)
X_tfidf = tv.fit_transform(df["tokens_filtered"])
feature_names = tv.get_feature_names_out()

dictionary = Dictionary(df["tokens_filtered"])
corpus = [dictionary.doc2bow(toks) for toks in df["tokens_filtered"]]

assert X_tfidf.shape[0] == len(df)
assert len(dictionary) == X_tfidf.shape[1]
print(df.shape, X_tfidf.shape, len(dictionary))

(19807, 6) (19807, 3571) 3571


In [9]:
# Finale Modelle trainieren 
start = time.perf_counter()
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=K_LDA,
               random_state=RANDOM_STATE, passes=LDA_PASSES)
t_lda = time.perf_counter() - start

start = time.perf_counter()
nmf = NMF(n_components=K_NMF, random_state=RANDOM_STATE, max_iter=NMF_MAX_ITER)
W = nmf.fit_transform(X_tfidf)
t_nmf = time.perf_counter() - start

assert nmf.n_iter_ < NMF_MAX_ITER, "NMF nicht konvergiert, NMF_MAX_ITER erhöhen"
print(f"LDA: {t_lda:.0f} s | NMF: {t_nmf:.1f} s ({nmf.n_iter_} Iterationen)")

LDA: 99 s | NMF: 6.4 s (217 Iterationen)


In [10]:
lda_top = [[w for w, _ in lda.show_topic(i, topn=TOPN)] for i in range(K_LDA)]
nmf_top = [[feature_names[j] for j in t.argsort()[-TOPN:][::-1]] for t in nmf.components_]

# Kontrolle: stimmen zwei Themen mit der Ausgabe aus 05 überein
erwartet_lda = {"mortgage", "loan", "home", "modification", "property",
                "tax", "foreclosure", "escrow", "insurance", "house"}
erwartet_nmf = {"loan", "student", "interest", "rate", "year",
                "school", "apply", "repayment", "private", "income"}
print("LDA Thema 12 wie in 05:", set(lda_top[11]) == erwartet_lda)
print("NMF Thema 7 wie in 05:", set(nmf_top[6]) == erwartet_nmf)

LDA Thema 12 wie in 05: True
NMF Thema 7 wie in 05: True


**Kontrolle:** 
Die neu trainierten Modelle liefern dieselben Themen wie in `05` (geprüft an LDA Thema 12 und NMF Thema 7). Die Modelle sind mit festem `random_state` reproduzierbar.

In [11]:
df["topic_lda"] = [max(t, key=lambda x: x[1])[0]
                   for t in lda.get_document_topics(corpus, minimum_probability=0)]
df["topic_nmf"] = W.argmax(axis=1)

In [12]:
pd.set_option("display.max_colwidth", None)   # Produktnamen nicht abkürzen
pd.set_option("display.width", 250)

def uebersicht(spalte, themen):
    anteil = df[spalte].value_counts(normalize=True)
    produkt = df.groupby(spalte)["Product"].agg(
        lambda s: ", ".join(f"{p} ({a:.0%})" for p, a in s.value_counts(normalize=True).head(2).items())
    )
    return pd.DataFrame({
        "Thema": [i + 1 for i in anteil.index],
        "Anteil %": (anteil.values * 100).round(1),
        "Top-Wörter": [", ".join(themen[i][:8]) for i in anteil.index],
        "häufigste Produkte": [produkt[i] for i in anteil.index],
    })

In [13]:
uebersicht("topic_lda", lda_top)

,Thema,Anteil %,Top-Wörter,häufigste Produkte
0,4,17.2,"report, credit, account, dispute, remove, information, file, reporting","Credit reporting, credit repair services, or other personal consumer reports (51%), Credit reporting (21%)"
1,11,12.6,"account, card, bank, check, charge, money, fee, close","Bank account or service (24%), Checking or savings account (22%)"
2,2,10.8,"tell, ask, phone, number, time, speak, contact, information","Debt collection (48%), Mortgage (9%)"
3,1,10.3,"payment, pay, receive, month, late, account, fee, statement","Mortgage (26%), Debt collection (14%)"
4,6,9.5,"debt, collection, letter, owe, company, send, receive, agency","Debt collection (81%), Credit reporting, credit repair services, or other personal consumer reports (10%)"
5,10,8.5,"loan, payment, pay, interest, year, month, rate, time","Student loan (50%), Mortgage (16%)"
6,7,7.9,"credit, report, inquiry, card, score, remove, company, authorize","Credit reporting, credit repair services, or other personal consumer reports (57%), Credit reporting (20%)"
7,12,6.1,"mortgage, loan, home, modification, property, tax, foreclosure, escrow","Mortgage (90%), Credit reporting, credit repair services, or other personal consumer reports (3%)"
8,8,5.9,"receive, request, letter, send, document, state, provide, email","Mortgage (43%), Debt collection (11%)"
9,5,5.4,"information, consumer, provide, law, request, file, verify, violation","Credit reporting, credit repair services, or other personal consumer reports (55%), Debt collection (15%)"


In [14]:
uebersicht("topic_nmf", nmf_top)

,Thema,Anteil %,Top-Wörter,häufigste Produkte
0,4,10.3,"report, credit, remove, bureau, score, item, reporting, collection","Credit reporting, credit repair services, or other personal consumer reports (51%), Credit reporting (23%)"
1,3,9.9,"debt, collection, collect, owe, collector, agency, original, company","Debt collection (82%), Credit reporting, credit repair services, or other personal consumer reports (9%)"
2,7,8.6,"loan, student, interest, rate, year, school, apply, repayment","Student loan (43%), Mortgage (22%)"
3,14,8.4,"information, dispute, verify, file, provide, request, consumer, delete","Credit reporting, credit repair services, or other personal consumer reports (53%), Credit reporting (19%)"
4,13,8.4,"mortgage, modification, home, property, foreclosure, escrow, sale, tax","Mortgage (86%), Credit reporting, credit repair services, or other personal consumer reports (5%)"
5,2,8.1,"payment, late, month, monthly, statement, day, date, time","Mortgage (23%), Credit reporting, credit repair services, or other personal consumer reports (16%)"
6,5,7.5,"tell, time, ask, speak, contact, work, day, try","Debt collection (28%), Mortgage (14%)"
7,6,6.3,"card, credit, purchase, score, apply, close, use, new","Credit card or prepaid card (33%), Credit card (30%)"
8,12,6.1,"check, bank, money, deposit, fund, transaction, cash, hold","Checking or savings account (26%), Bank account or service (25%)"
9,10,5.7,"charge, fee, interest, overdraft, late, service, statement, balance","Credit card or prepaid card (18%), Credit card (15%)"


In [15]:
overlap = pd.DataFrame(
    [[len(set(a) & set(b)) for b in nmf_top] for a in lda_top],
    index=[f"LDA {i + 1}" for i in range(K_LDA)],
    columns=[f"NMF {j + 1}" for j in range(K_NMF)],
)
aehnlichstes = pd.DataFrame({
    "ähnlichstes NMF-Thema": overlap.idxmax(axis=1),
    "gemeinsame Top-Wörter": overlap.max(axis=1),
})
print("Wörter in mehreren Themen:",
      "LDA", sum(c > 1 for c in Counter(w for t in lda_top for w in t).values()),
      "| NMF", sum(c > 1 for c in Counter(w for t in nmf_top for w in t).values()))
aehnlichstes

Wörter in mehreren Themen: LDA 24 | NMF 25


,ähnlichstes NMF-Thema,gemeinsame Top-Wörter
LDA 1,NMF 2,5
LDA 2,NMF 5,6
LDA 3,NMF 15,3
LDA 4,NMF 4,5
LDA 5,NMF 14,6
LDA 6,NMF 3,6
LDA 7,NMF 8,6
LDA 8,NMF 9,5
LDA 9,NMF 15,2
LDA 10,NMF 7,6


## Diskussion

**Ergebnis:**
Beide Verfahren finden dieselben Kernbereiche der Beschwerden. Fasst man ähnliche Themen zusammen, ist die Kreditauskunft der größte Bereich mit rund 23 % (NMF) bis 31 % (LDA) der Beschwerden, danach folgen Inkasso, Hypothek, Studienkredit und Bankkonto. Für 10 von 12 LDA-Themen gibt es ein NMF-Thema mit mindestens 5 gemeinsamen Top-Wörtern. Mehrere Themen passen eindeutig zu den Produktkategorien der Behörde (Hypothek 86 bis 90 %, Inkasso 81 bis 82 %).

**Grenzen:**
- Einige Themen beschreiben Handlungen und kommen bei vielen Produkten vor (Zahlungen, Schriftverkehr, Gebühren) und lassen sich keinem Produkt zuordnen.
- In beiden Modellen bleibt ein Thema aus Erzählwörtern, obwohl TF-IDF häufige Wörter dämpft.
- Die Anteile beruhen auf dem dominanten Thema je Beschwerde, das ist eine Vereinfachung.
- LDA und NMF unterscheiden sich in Eingabe und Themenzahl, Unterschiede sind nicht eindeutig zuordenbar.

